In [10]:
import glob
import pandas as pd
from helper import *
from torch.optim import Adam
from torchinfo import summary
from torch.utils.data import Dataset, DataLoader

I moved classes and functions from the previous notebook to the `helper.py` not to rewrite the same code

In [11]:
HR_train_paths = sorted(glob.glob("../data/DIV2K_train_HR/*.png"))
X2_train_paths = sorted(glob.glob("../data/DIV2K_train_LR_bicubic/X2/*.png"))
X4_train_paths = sorted(glob.glob("../data/DIV2K_train_LR_bicubic/X4/*.png"))
X8_train_paths = sorted(glob.glob("../data/DIV2K_train_LR_bicubic/X8/*.png"))
X16_train_paths = sorted(glob.glob("../data/DIV2K_train_LR_bicubic/X16/*.png"))
X32_train_paths = sorted(glob.glob("../data/DIV2K_train_LR_bicubic/X32/*.png"))
X64_train_paths = sorted(glob.glob("../data/DIV2K_train_LR_bicubic/X64/*.png"))

HR_valid_paths = sorted(glob.glob("../data/DIV2K_valid_HR/*.png"))
X2_valid_paths = sorted(glob.glob("../data/DIV2K_valid_LR_bicubic/X2/*.png"))
X4_valid_paths = sorted(glob.glob("../data/DIV2K_valid_LR_bicubic/X4/*.png"))
X8_valid_paths = sorted(glob.glob("../data/DIV2K_valid_LR_bicubic/X8/*.png"))
X16_valid_paths = sorted(glob.glob("../data/DIV2K_valid_LR_bicubic/X16/*.png"))
X32_valid_paths = sorted(glob.glob("../data/DIV2K_valid_LR_bicubic/X32/*.png"))
X64_valid_paths = sorted(glob.glob("../data/DIV2K_valid_LR_bicubic/X64/*.png"))

## X4 Scaling

In [12]:
# checkpoint = torch.load('./model_checkpoints/EDSR/X2.pth')
model = EDSR_Light(2).to(device)
# model.load_state_dict(checkpoint['model_state_dict'])
model.upscaling_head

Sequential(
  (0): Conv2d(64, 256, kernel_size=(3, 3), stride=(1, 1), padding=same)
  (1): PixelShuffle(upscale_factor=2)
  (2): PReLU(num_parameters=1)
  (3): Conv2d(64, 3, kernel_size=(9, 9), stride=(1, 1), padding=same)
)

In [4]:
# remove the last convolutional layer
model.upscaling_head = model.upscaling_head[:-1]
model.upscaling_head

Sequential(
  (0): Conv2d(64, 256, kernel_size=(3, 3), stride=(1, 1), padding=same)
  (1): PixelShuffle(upscale_factor=2)
  (2): PReLU(num_parameters=1)
)

In [5]:
head = nn.Sequential(
    nn.Conv2d(64, 256, 3, stride=1, padding='same'),
    nn.PixelShuffle(2),
    nn.PReLU(),
    nn.Conv2d(64, 3, 9, stride=1, padding='same')
)

In [6]:
model.upscaling_head = nn.Sequential(*model.upscaling_head, *head)
model.upscaling_head

Sequential(
  (0): Conv2d(64, 256, kernel_size=(3, 3), stride=(1, 1), padding=same)
  (1): PixelShuffle(upscale_factor=2)
  (2): PReLU(num_parameters=1)
  (3): Conv2d(64, 256, kernel_size=(3, 3), stride=(1, 1), padding=same)
  (4): PixelShuffle(upscale_factor=2)
  (5): PReLU(num_parameters=1)
  (6): Conv2d(64, 3, kernel_size=(9, 9), stride=(1, 1), padding=same)
)

In [7]:
model.upscaling_head[:-4]

Sequential(
  (0): Conv2d(64, 256, kernel_size=(3, 3), stride=(1, 1), padding=same)
  (1): PixelShuffle(upscale_factor=2)
  (2): PReLU(num_parameters=1)
)

For the first 1000 epochs I freeze the pretrained layers

In [8]:
for param in model.expand.parameters():
    param.requires_grad = False

for param in model.residual_blocks.parameters():
    param.requires_grad = False

for param in model.upscaling_head[:-4].parameters():
    param.requires_grad = False

In [9]:
model.upscaling_head[-4:]

Sequential(
  (3): Conv2d(64, 256, kernel_size=(3, 3), stride=(1, 1), padding=same)
  (4): PixelShuffle(upscale_factor=2)
  (5): PReLU(num_parameters=1)
  (6): Conv2d(64, 3, kernel_size=(9, 9), stride=(1, 1), padding=same)
)

In [62]:
summary(model, col_names=['trainable'])

In [ ]:
valid_ds = EDSR_Dataset(HR_valid_paths, 4, ram_limit_gb=1)

In [ ]:
train_ds = EDSR_Dataset(HR_train_paths, 4, ram_limit_gb=8)

In [ ]:
train_dl = DataLoader(train_ds, batch_size=16, shuffle=True, num_workers=os.cpu_count()-1)
valid_dl = DataLoader(valid_ds, batch_size=16, shuffle=False, num_workers=os.cpu_count()-1)

loss_fn = nn.L1Loss()
optimizer = Adam(model.upscaling_head[-4:].parameters(), lr=1e-4)
scheduler = StepLR(optimizer, step_size=200, gamma=0.5)
model = troch.compile(model)

train(model, train_dl, valid_dl, optimizer, scheduler, loss_fn, 1000)

In [ ]:
for param in model.expand.parameters():
    param.requires_grad = True

for param in model.residual_blocks.parameters():
    param.requires_grad = True

for param in model.upscaling_head[:-4].parameters():
    param.requires_grad = True

In [ ]:
optimizer = Adam(model.parameters(), lr=1e-5)
scheduler = StepLR(optimizer, step_size=1000, gamma=0.5)

train(model, train_dl, valid_dl, optimizer, scheduler, loss_fn, 5000)

In [ ]:
model.eval()

### Super-resolution showcase

## Geometric Self-ensemble X4

In [ ]:
class GSE:
    def __init__(self, model):
        self.model = model
        self.rotations = [0, 90, 180, 270]
        
        self.DIV2K_RGB = torch.tensor([0.44882884613943946, 0.43713809810624193, 0.4040371984052683], device='cpu')

        self.transforms = v2.Compose([
            v2.PILToTensor(),
            v2.Lambda(lambda x: (x / 255.0) - self.DIV2K_RGB[:, None, None])
        ])

    def __call__(self, x):
        prediction = torch.zeros(self.model(x).shape).to(device)
        for rotation in self.rotations:
            rot = torch.rot90(x, k=rotation // 90, dims=[-2, -1])
            for i in range(2):
                with torch.inference_mode():
                    if i:
                        flip = v2.functional.horizontal_flip(rot)
                        out = torch.flip(self.model(flip), dims=[-1])
                    else:
                        out = self.model(rot)

                prediction += torch.rot90(out, k=4 - rotation // 90, dims=[-2, -1])

        return prediction / 8.0

In [ ]:
model_gse = GSE(model)

### Super-resolution showcase

## X8 Scaling

The process of preparation for training is the same as above, simply the starting point is the **4X model**, not the 2X model

In [64]:
# checkpoint = torch.load('./model_checkpoints/EDSR/X4.pth')
model = EDSR_Light(4).to(device)
# model.load_state_dict(checkpoint['model_state_dict'])

model.upscaling_head = model.upscaling_head[:-1]

head = nn.Sequential(
    nn.Conv2d(64, 256, 3, stride=1, padding='same'),
    nn.PixelShuffle(2),
    nn.PReLU(),
    nn.Conv2d(64, 3, 9, stride=1, padding='same')
)

model.upscaling_head = nn.Sequential(*model.upscaling_head, *head)

for param in model.expand.parameters():
    param.requires_grad = False

for param in model.residual_blocks.parameters():
    param.requires_grad = False

for param in model.upscaling_head[:-4].parameters():
    param.requires_grad = False

summary(model, col_names=['trainable'])

Layer (type:depth-idx)                   Trainable
EDSR_Light                               Partial
├─Sequential: 1-1                        False
│    └─Conv2d: 2-1                       False
│    └─PReLU: 2-2                        False
├─Sequential: 1-2                        False
│    └─ResBlockEDSRLight: 2-3            False
│    │    └─Sequential: 3-1              False
│    └─ResBlockEDSRLight: 2-4            False
│    │    └─Sequential: 3-2              False
│    └─ResBlockEDSRLight: 2-5            False
│    │    └─Sequential: 3-3              False
│    └─ResBlockEDSRLight: 2-6            False
│    │    └─Sequential: 3-4              False
│    └─ResBlockEDSRLight: 2-7            False
│    │    └─Sequential: 3-5              False
│    └─ResBlockEDSRLight: 2-8            False
│    │    └─Sequential: 3-6              False
│    └─ResBlockEDSRLight: 2-9            False
│    │    └─Sequential: 3-7              False
│    └─ResBlockEDSRLight: 2-10           False
│    │ 

In [ ]:
valid_ds = EDSR_Dataset(HR_valid_paths, 2, ram_limit_gb=1)

In [ ]:
train_ds = EDSR_Dataset(HR_train_paths, 2, ram_limit_gb=8)

In [ ]:
train_dl = DataLoader(train_ds, batch_size=16, shuffle=True, num_workers=os.cpu_count()-1)
valid_dl = DataLoader(valid_ds, batch_size=16, shuffle=False, num_workers=os.cpu_count()-1)

loss_fn = nn.L1Loss()
optimizer = Adam(model.upscaling_head[-4:].parameters(), lr=1e-4)
scheduler = StepLR(optimizer, step_size=200, gamma=0.5)
model = torch.compile(model)

train(model, train_dl, valid_dl, optimizer, scheduler, loss_fn, 1000)

In [ ]:
for param in model.expand.parameters():
    param.requires_grad = True

for param in model.residual_blocks.parameters():
    param.requires_grad = True

for param in model.upscaling_head[:-4].parameters():
    param.requires_grad = True

In [ ]:
optimizer = Adam(model.parameters(), lr=1e-5)
scheduler = StepLR(optimizer, step_size=1000, gamma=0.5)

train(model, train_dl, valid_dl, optimizer, scheduler, loss_fn, 5000)

In [ ]:
model.eval()

### Super-resolution showcase

## Geometric Self-ensemble X8

In [1]:
class GSE:
    def __init__(self, model):
        self.model = model
        self.rotations = [0, 90, 180, 270]
        
        self.DIV2K_RGB = torch.tensor([0.44882884613943946, 0.43713809810624193, 0.4040371984052683], device='cpu')

        self.transforms = v2.Compose([
            v2.PILToTensor(),
            v2.Lambda(lambda x: (x / 255.0) - self.DIV2K_RGB[:, None, None])
        ])

    def __call__(self, x):
        prediction = torch.zeros(self.model(x).shape).to(device)
        for rotation in self.rotations:
            rot = torch.rot90(x, k=rotation // 90, dims=[-2, -1])
            for i in range(2):
                with torch.inference_mode():
                    if i:
                        flip = v2.functional.horizontal_flip(rot)
                        out = torch.flip(self.model(flip), dims=[-1])
                    else:
                        out = self.model(rot)

                prediction += torch.rot90(out, k=4 - rotation // 90, dims=[-2, -1])

        return prediction / 8.0

In [ ]:
model_gse = GSE(model)

### Super-resolution showcase